# GymRec: learn correct form from the dataset (form grading)

<a href="https://colab.research.google.com/github/Sarthak-D96/Gym_Exercise/blob/main/training/colab_form_profiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The dataset ([Gym Exercise DataSet, 68K frames](https://www.kaggle.com/datasets/amanyadav2030a/processed-train-test-exercise-68k),
by Aman Yadav, CC0) has no "good form / bad form" labels, but its videos demonstrate each exercise.
This notebook replays every training video through the website's own rep counter and form measurements and learns,
per exercise, what normal reps look like:

* **rep-counting angles** calibrated from the videos (instead of hand-set guesses);
* the normal range of **depth, lockout, tempo, torso lean, hip line, elbow drift, left/right symmetry**.

On the website every rep is graded against these ranges: **good** (counted), **minor** (counted, with a tip),
**major** (not counted, with a message saying what and by how much). The held-out test videos show how many reps
would be counted. They are demonstrations, so that share should be high.

**Fast:** it uses the keypoints already in Google Drive from the training notebooks (no GPU or retraining needed).
Without them, it downloads the dataset and extracts keypoints first (needs a GPU runtime and the Kaggle secrets,
see `colab_two_stage.ipynb`).

In [ ]:
# 1. Google Drive (same folder as the training notebooks)
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/gymrec/work"

In [ ]:
# 2. Get the code
%cd /content
!test -d Gym_Exercise || git clone -q https://github.com/Sarthak-D96/Gym_Exercise.git
%cd /content/Gym_Exercise
!git pull -q
!test -f training/build_form_profiles.py || echo "!! This copy of the repo has no form profiles code: push it to GitHub first."

In [ ]:
# 3. Keypoints already extracted? Otherwise download the dataset and extract them.
from pathlib import Path
have = len(list(Path(WORK_DIR, "keypoints").rglob("*.npz")))
print(f"{have} keypoint files in Drive")
NEED_EXTRACT = have == 0
TRAIN_DIR = TEST_DIR = None
if NEED_EXTRACT:
    import os
    from google.colab import userdata
    !pip install -q ultralytics kagglehub
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ[k] = userdata.get(k)
    import kagglehub
    root = Path(kagglehub.dataset_download("amanyadav2030a/processed-train-test-exercise-68k"))
    find = lambda n: max((p for p in root.rglob(n) if p.is_dir()), key=lambda p: len(p.parts), default=None)
    TRAIN_DIR, TEST_DIR = find("processed-train"), find("processed-test")
    print("train:", TRAIN_DIR, "\ntest: ", TEST_DIR)

In [ ]:
# 4. Configure (same classes as the model you trained)
import sys; sys.path.insert(0, "/content/Gym_Exercise/training")
import run_all
config = run_all.config
config.CLASSES = [c for c in config.EXERCISES if c not in ("lunge", "jumping_jack")]   # all 22 dataset classes
config.WORK_DIR = WORK_DIR
config.FPS = 5          # the rate the keypoints were extracted at
config.FRAMES_FPS = 5
if NEED_EXTRACT:
    config.DATA_DIRS = [str(TRAIN_DIR)]
    config.TEST_DIRS = [str(TEST_DIR)] if TEST_DIR else []

In [ ]:
# 5. Learn the form profiles (a few minutes)
run_all.main(["extract", "profiles"] if NEED_EXTRACT else ["profiles"])

In [ ]:
# 6. Summary and download
import json
from google.colab import files

prof = json.load(open(f"{WORK_DIR}/form_profiles.json"))["exercises"]
print(f"{'exercise':22s} {'reps':>5s}  {'rep angles':>18s}  held-out reps: good / minor / not counted")
for slug, e in prof.items():
    a, t = e["angles"], e.get("test", {})
    ang = f"{a['low']:.0f}–{a['high']:.0f}°" + (" (calibrated)" if a.get("applied") else " (kept)")
    print(f"{slug:22s} {e['reps']:5d}  {ang:>18s}  "
          f"{t.get('good', 0)} / {t.get('minor', 0)} / {t.get('major', 0) + t.get('partial', 0)}")
files.download(f"{WORK_DIR}/form_profiles.json")

## 7. Use it on the website (on your laptop)

```bash
cd ~/Desktop/Gym_Exercise
source .venv/bin/activate
python manage.py migrate
python manage.py install_form_profiles ~/Downloads/form_profiles.json
python manage.py runserver
```

* `install_form_profiles` stores the normal ranges per exercise and switches the rep-counting angles to the calibrated
  ones (`--keep-angles` to keep yours).
* Model zips made by the training notebooks from now on contain `form_profiles.json`, and `install_model` installs it too.
* To count every full rep and only show form tips, set `RECOGNITION_COUNT_ONLY_GOOD_FORM=false` in `.env`.
* Too strict or too lenient for one exercise? In Django admin (Workouts → Exercises) you can clear its
  form profile to go back to the built-in defaults.

**Read the held-out column before trusting the grader.** Those reps are demonstrations. If many of an exercise's
reps are "not counted", the measurements don't fit that exercise's camera angle: say so in the journal and clear
its profile.